<a href="https://colab.research.google.com/github/FC-Andrade/Analises-complementares/blob/main/AN%C3%81LISE_DE_COMPLEMENTARIDADE_ELETROST%C3%81TICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ANÁLISE DE COMPLEMENTARIDADE ELETROSTÁTICA AUTOMATIZADA
# Suporta múltiplos arquivos PDB (triplicatas Apo e AG73)
# ============================================================

# 1. Instalação de pacotes necessários
!apt-get install -y apbs > /dev/null
!pip install biopython pymol-open-source > /dev/null

import os
from Bio.PDB import PDBParser
from google.colab import files
import pymol
from pymol import cmd

WORKDIR = "/content/electrostatics"
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)


In [ ]:
# 2. Upload de múltiplos arquivos PDB
print("📂 Faça upload de todos os arquivos PDB das triplicatas (Apo e AG73).")
uploaded = files.upload()
pdb_files = list(uploaded.keys())
print(f"{len(pdb_files)} arquivos carregados: {pdb_files}")

In [ ]:
# 3. Inicializa PyMOL em modo silencioso
pymol.finish_launching(['pymol', '-qc'])

# 4. Função para processamento de cada PDB
def run_electrostatics(pdb_file):
    base_name = pdb_file.replace(".pdb", "")
    print(f"Processando: {base_name}")

    # 3a. Conversão PDB → PQR
    !pdb2pqr30 --ff=PARSE {pdb_file} {base_name}.pqr

    # 4a. Criação do arquivo de entrada APBS
    apbs_input = f"""
read
    mol pqr {base_name}.pqr
end
elec
    mg-auto
    dime 129 129 129
    cglen 80 80 80
    fglen 80 80 80
    mol 1
    lpbe
    bcfl sdh
    pdie 2.0
    sdie 78.54
    srfm smol
    chgm spl2
    sdens 10.0
    srad 1.4
    swin 0.3
    temp 298.15
    calcenergy total
    calcforce no
    write pot dx {base_name}
end
quit
"""
    with open(f"{base_name}.in", "w") as f:
        f.write(apbs_input)

    !apbs {base_name}.in > {base_name}_apbs.log

    # 5a. Visualização e renderização em PyMOL
    cmd.load(f"{base_name}.pdb", base_name)
    cmd.load(f"{base_name}.dx", f"{base_name}_potential")
    cmd.show("surface", base_name)
    cmd.color("white", base_name)
    cmd.ramp_new(f"{base_name}_ramp", f"{base_name}_potential", [-2, 0, 2], ["red", "white", "blue"])
    cmd.set("surface_color", f"{base_name}_ramp", base_name)
    cmd.png(f"{base_name}_electrostatics.png", width=1600, height=1200, dpi=300, ray=1)

# 5. Processa todos os arquivos
for pdb in pdb_files:
    run_electrostatics(pdb)



In [ ]:
# 6. (Opcional) criar figura combinada com todos os sistemas carregados
cmd.bg_color("white")
for pdb in pdb_files:
    base_name = pdb.replace(".pdb", "")
    cmd.show("surface", base_name)
    cmd.set("transparency", 0.5, base_name)
cmd.png("Combined_Electrostatics.png", width=1800, height=1400, dpi=300, ray=1)

cmd.quit()

print("✅ Análise concluída.")
print("Imagens geradas para cada triplicata e 'Combined_Electrostatics.png'.")
